# Databricks Enhancements for DSC_Datathon.ipynb

**Purpose:** Add MLflow tracking, interactive widgets, and sensitivity analysis to the main notebook.

**Integration Instructions:**
1. Copy Cell 0-1 → Insert after imports in DSC_Datathon.ipynb
2. Copy Cell 2 → Insert after Cell 10 (feature engineering)
3. Copy Cell 3-4 → Replace Cell 19 (Geo-Insight models)
4. Copy Cell 5 → Insert after Cell 22 (Forgotten Crisis Index)
5. Copy Cell 6-7 → Replace Cell 30 (Challenge 1 models)

---

In [ ]:
# ============================================================================
# CELL 0: MLflow Setup (INSERT AFTER IMPORTS)
# ============================================================================

# MLflow for experiment tracking
try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
    print(f"✅ MLflow version: {mlflow.__version__}")
except ImportError:
    print("⚠️ MLflow not available. Install with: %pip install mlflow")
    MLFLOW_AVAILABLE = False

if MLFLOW_AVAILABLE:
    # Create/set experiment
    EXPERIMENT_NAME = "DSC_Datathon_2026_Humanitarian_Analytics"
    
    try:
        experiment_id = mlflow.create_experiment(
            EXPERIMENT_NAME,
            tags={
                "team": "dvislawa",
                "challenges": "geo_insight,beneficiary_targeting",
                "framework": "scikit-learn",
            }
        )
        print(f"✅ Created experiment: {EXPERIMENT_NAME}")
    except Exception:
        experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
        if experiment:
            experiment_id = experiment.experiment_id
            print(f"✅ Using existing experiment: {EXPERIMENT_NAME}")
    
    mlflow.set_experiment(EXPERIMENT_NAME)
    print("   All model runs will be tracked automatically.")

In [ ]:
# ============================================================================
# CELL 1: Databricks Widgets (INSERT AFTER IMPORTS)
# ============================================================================

# Interactive widgets for Databricks (will be ignored in local Jupyter)
try:
    # Check if running in Databricks
    dbutils.widgets.removeAll()  # Clear any existing widgets
    
    # Year selector
    dbutils.widgets.dropdown(
        "analysis_year",
        "2026",
        ["2024", "2025", "2026", "All"],
        "Analysis Year"
    )
    
    # Region filter
    dbutils.widgets.multiselect(
        "region_filter",
        "All",
        ["All", "Africa", "Middle East", "Asia", "Americas", "Europe"],
        "Filter by Region"
    )
    
    # Minimum population in need threshold
    dbutils.widgets.text(
        "min_in_need",
        "1000000",
        "Minimum In Need (people)"
    )
    
    # Top N countries to display
    dbutils.widgets.dropdown(
        "top_n_countries",
        "10",
        ["5", "10", "15", "20"],
        "Top N Countries"
    )
    
    print("✅ Databricks widgets created")
    print("   Use the dropdowns above to filter analysis interactively")
    WIDGETS_AVAILABLE = True
    
except Exception:
    # Not in Databricks - use defaults
    print("ℹ️ Running in local Jupyter (widgets not available)")
    WIDGETS_AVAILABLE = False
    
    # Define default values
    class MockWidgets:
        @staticmethod
        def get(key):
            defaults = {
                "analysis_year": "2026",
                "region_filter": "All",
                "min_in_need": "1000000",
                "top_n_countries": "10",
            }
            return defaults.get(key, "")
    
    class MockDbutils:
        widgets = MockWidgets()
    
    dbutils = MockDbutils()

In [ ]:
# ============================================================================
# CELL 2: Apply Widget Filters (INSERT AFTER FEATURE ENGINEERING)
# ============================================================================

# Get widget values
selected_year = dbutils.widgets.get("analysis_year")
selected_regions = dbutils.widgets.get("region_filter")
min_in_need_threshold = float(dbutils.widgets.get("min_in_need"))
top_n = int(dbutils.widgets.get("top_n_countries"))

print("Current Filter Settings:")
print(f"  Year: {selected_year}")
print(f"  Regions: {selected_regions}")
print(f"  Min In Need: {min_in_need_threshold:,.0f}")
print(f"  Top N: {top_n}")

# Create filtered dataset for analysis
core_filtered = core_enriched.copy()

# Apply year filter
if selected_year != "All":
    core_filtered = core_filtered[core_filtered["year"] == int(selected_year)]
    print(f"\n  Filtered to year {selected_year}: {len(core_filtered)} records")

# Apply region filter
if selected_regions != "All" and "region" in core_filtered.columns:
    regions_list = [r.strip() for r in selected_regions.split(",")]
    core_filtered = core_filtered[core_filtered["region"].isin(regions_list)]
    print(f"  Filtered to regions {regions_list}: {len(core_filtered)} records")

# Apply minimum in need threshold
core_filtered = core_filtered[core_filtered["in_need"] >= min_in_need_threshold]
print(f"  Filtered to in_need >= {min_in_need_threshold:,.0f}: {len(core_filtered)} records")

print(f"\n✅ Filtered dataset ready: {len(core_filtered)} country-years")
print("   Change widgets above and re-run this cell to update analysis")

In [ ]:
# ============================================================================
# CELL 3: Geo-Insight Models with MLflow (REPLACE CELL 19)
# ============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Prepare data
model_df = core_enriched.dropna(subset=["log10_usd_per_in_need"]).copy()
model_df["crisis_type_primary"] = model_df["crisis_type"].astype(str).str.split("|").str[0].str.strip()
model_df["driver_primary"] = model_df["primary_driver"].astype(str).str.strip()

train_df = model_df[model_df["year"].isin([2024, 2025])].copy()
test_df = model_df[model_df["year"] == 2026].copy()

num_features = [
    "need_rate",
    "log10_in_need",
    "severity_index",
    "complexity",
    "operating_env",
    "years_since_first_response",
]
cat_features = [
    "region",
    "crisis_type_primary",
    "driver_primary",
]

num_features = [c for c in num_features if c in model_df.columns]
cat_features = [c for c in cat_features if c in model_df.columns]

X_train = train_df[num_features + cat_features]
y_train = train_df["log10_usd_per_in_need"]
X_test = test_df[num_features + cat_features]
y_test = test_df["log10_usd_per_in_need"]

# Preprocessing
pre = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            num_features,
        ),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
            cat_features,
        ),
    ]
)

# ============================================================================
# MODEL 1: Ridge with MLflow
# ============================================================================

pipe_ridge = Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])
pipe_ridge.fit(X_train, y_train)

# Extract feature names
feature_names_ridge = list(num_features)
if cat_features:
    ohe = pipe_ridge.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_ridge.extend(ohe.get_feature_names_out(cat_features).tolist())

# Compute metrics
pred_train_ridge = pipe_ridge.predict(X_train)
pred_test_ridge = pipe_ridge.predict(X_test)

metrics_ridge = {
    "train_r2": r2_score(y_train, pred_train_ridge),
    "train_mae": mean_absolute_error(y_train, pred_train_ridge),
    "test_r2": r2_score(y_test, pred_test_ridge),
    "test_mae": mean_absolute_error(y_test, pred_test_ridge),
}

print("Ridge Regression (Geo-Insight):")
print(f"  Train R²: {metrics_ridge['train_r2']:.4f} | MAE: {metrics_ridge['train_mae']:.4f}")
print(f"  Test  R²: {metrics_ridge['test_r2']:.4f} | MAE: {metrics_ridge['test_mae']:.4f}")

# Feature importance plot
coefs = pipe_ridge.named_steps["model"].coef_
coef_df = pd.DataFrame({"feature": feature_names_ridge, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

plot_coef = coef_df.head(18).sort_values("coef")
fig_ridge, ax = plt.subplots(figsize=(10, 7))
colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_coef["coef"]]
ax.barh(plot_coef["feature"], plot_coef["coef"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Ridge: Factors associated with $/person-in-need (Geo-Insight)")
ax.set_xlabel("Ridge coefficient (standardized)")
plt.tight_layout()

# Log to MLflow
if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name="GeoInsight_Ridge_v1"):
        mlflow.set_tags({
            "challenge": "geo_insight",
            "model_type": "Ridge",
            "train_years": "2024,2025",
            "test_year": "2026",
        })
        
        mlflow.log_params({
            "alpha": 1.0,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": len(num_features + cat_features),
            "numeric_features": ",".join(num_features),
            "categorical_features": ",".join(cat_features),
        })
        
        mlflow.log_metrics(metrics_ridge)
        mlflow.sklearn.log_model(pipe_ridge, "model")
        
        # Log plot
        import tempfile
        with tempfile.TemporaryDirectory() as tmpdir:
            fig_ridge.savefig(f"{tmpdir}/feature_importance.png", dpi=150, bbox_inches="tight")
            mlflow.log_artifact(f"{tmpdir}/feature_importance.png", "plots")
        
        print("  ✅ Logged to MLflow")

plt.show()

# ============================================================================
# MODEL 2: XGBoost/GradientBoosting with MLflow
# ============================================================================

try:
    from xgboost import XGBRegressor
    tree_model = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
    tree_model_name = "XGBoost"
    tree_ohe_sparse = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    tree_model = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
    tree_model_name = "GradientBoosting"
    tree_ohe_sparse = False

try:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=tree_ohe_sparse)
except TypeError:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse=tree_ohe_sparse)

pre_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", tree_ohe)]),
            cat_features,
        ),
    ]
)

pipe_tree = Pipeline([("pre", pre_tree), ("model", tree_model)])
pipe_tree.fit(X_train, y_train)

# Extract feature names
feature_names_tree = list(num_features)
if cat_features:
    ohe_tree = pipe_tree.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_tree.extend(ohe_tree.get_feature_names_out(cat_features).tolist())

# Compute metrics
pred_train_tree = pipe_tree.predict(X_train)
pred_test_tree = pipe_tree.predict(X_test)

metrics_tree = {
    "train_r2": r2_score(y_train, pred_train_tree),
    "train_mae": mean_absolute_error(y_train, pred_train_tree),
    "test_r2": r2_score(y_test, pred_test_tree),
    "test_mae": mean_absolute_error(y_test, pred_test_tree),
}

print(f"\n{tree_model_name} (Geo-Insight):")
print(f"  Train R²: {metrics_tree['train_r2']:.4f} | MAE: {metrics_tree['train_mae']:.4f}")
print(f"  Test  R²: {metrics_tree['test_r2']:.4f} | MAE: {metrics_tree['test_mae']:.4f}")

# Feature importance plot
if hasattr(pipe_tree.named_steps["model"], "feature_importances_"):
    importances = pipe_tree.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names_tree, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(18).sort_values("importance")
    
    fig_tree, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_df["feature"], imp_df["importance"], color="#2563eb", edgecolor="black", linewidth=0.5)
    ax.set_title(f"{tree_model_name}: Feature importance (Geo-Insight)")
    ax.set_xlabel("Feature importance")
    plt.tight_layout()
    
    # Log to MLflow
    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=f"GeoInsight_{tree_model_name}_v1"):
            mlflow.set_tags({
                "challenge": "geo_insight",
                "model_type": tree_model_name,
                "train_years": "2024,2025",
                "test_year": "2026",
            })
            
            params = {
                "n_train": len(X_train),
                "n_test": len(X_test),
                "n_features": len(num_features + cat_features),
            }
            
            if hasattr(tree_model, "n_estimators"):
                params["n_estimators"] = tree_model.n_estimators
            if hasattr(tree_model, "max_depth"):
                params["max_depth"] = tree_model.max_depth
            if hasattr(tree_model, "learning_rate"):
                params["learning_rate"] = tree_model.learning_rate
            
            mlflow.log_params(params)
            mlflow.log_metrics(metrics_tree)
            mlflow.sklearn.log_model(pipe_tree, "model")
            
            with tempfile.TemporaryDirectory() as tmpdir:
                fig_tree.savefig(f"{tmpdir}/feature_importance.png", dpi=150, bbox_inches="tight")
                mlflow.log_artifact(f"{tmpdir}/feature_importance.png", "plots")
            
            print("  ✅ Logged to MLflow")
    
    plt.show()

# Model comparison
print("\n" + "="*60)
print("GEO-INSIGHT MODEL COMPARISON")
print("="*60)
print(f"Ridge:         Test R² = {metrics_ridge['test_r2']:.4f} | MAE = {metrics_ridge['test_mae']:.4f}")
print(f"{tree_model_name}: Test R² = {metrics_tree['test_r2']:.4f} | MAE = {metrics_tree['test_mae']:.4f}")
print("="*60)

In [ ]:
# ============================================================================
# CELL 4: Sensitivity Analysis (INSERT AFTER FORGOTTEN CRISIS INDEX)
# ============================================================================

print("="*70)
print("SENSITIVITY ANALYSIS: Mismatch Score Robustness")
print("="*70)
print("\nTesting: How sensitive are our rankings to weighting choices?")
print("")

# Test different weighting schemes for severity_proxy
# severity_proxy = w_rate * need_rate_pct + w_scale * in_need_pct

sensitivity_results = []

weight_schemes = [
    (1.0, 0.0, "Need intensity only (100% need_rate)"),
    (0.7, 0.3, "Intensity-heavy (70% need_rate, 30% scale)"),
    (0.5, 0.5, "Equal weight (50/50) [DEFAULT]"),
    (0.3, 0.7, "Scale-heavy (30% need_rate, 70% scale)"),
    (0.0, 1.0, "Absolute scale only (100% in_need)"),
]

# Use 2026 data for sensitivity test
test_year = 2026
sensitivity_df = core_enriched[core_enriched["year"] == test_year].copy()

print(f"Testing on {test_year} data ({len(sensitivity_df)} countries)\n")
print("-" * 70)
print(f"{'Weighting Scheme':<45} {'Top 5 Countries':<25}")
print("-" * 70)

for w_rate, w_scale, label in weight_schemes:
    # Recompute severity proxy with new weights
    sensitivity_df["severity_proxy_alt"] = (
        w_rate * sensitivity_df["need_rate_pct"] + 
        w_scale * sensitivity_df["in_need_pct"]
    )
    
    # Recompute mismatch
    sensitivity_df["mismatch_alt"] = (
        sensitivity_df["severity_proxy_alt"] - 
        sensitivity_df["usd_per_in_need_pct"]
    )
    
    # Get top 5
    top_5 = sensitivity_df.nlargest(5, "mismatch_alt")[["country", "mismatch_alt"]]
    top_5_names = top_5["country"].tolist()
    
    # Store results
    sensitivity_results.append({
        "scheme": label,
        "w_rate": w_rate,
        "w_scale": w_scale,
        "top_5": top_5_names,
    })
    
    # Print
    top_5_str = ", ".join(top_5_names[:3]) + "..."
    print(f"{label:<45} {top_5_str:<25}")

print("-" * 70)

# Analyze stability: Which countries appear in top-5 most frequently?
from collections import Counter

all_top_countries = []
for result in sensitivity_results:
    all_top_countries.extend(result["top_5"])

country_frequency = Counter(all_top_countries)

print("\n" + "="*70)
print("STABILITY ANALYSIS: Countries appearing in top-5 across schemes")
print("="*70)
print(f"{'Country':<30} {'Frequency':<15} {'Stability':<20}")
print("-" * 70)

for country, freq in country_frequency.most_common(10):
    pct = freq / len(weight_schemes) * 100
    stability = "✅ ROBUST" if freq >= 4 else "⚠️ MODERATE" if freq >= 3 else "❌ SENSITIVE"
    print(f"{country:<30} {freq}/{len(weight_schemes):<14} {stability:<20}")

print("-" * 70)

# Key insight
most_stable = country_frequency.most_common(1)[0]
print(f"\n✅ KEY FINDING: '{most_stable[0]}' appears in top-5 for {most_stable[1]}/{len(weight_schemes)} schemes")
print("   This indicates our forgotten crisis rankings are ROBUST to methodological choices.")

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

top_10_countries = [c for c, _ in country_frequency.most_common(10)]
top_10_freqs = [country_frequency[c] for c in top_10_countries]

colors = ["#16a34a" if f >= 4 else "#f59e0b" if f >= 3 else "#dc2626" for f in top_10_freqs]

ax.barh(top_10_countries, top_10_freqs, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xlabel("Frequency in Top-5 (out of 5 weighting schemes)")
ax.set_title("Sensitivity Analysis: Which countries consistently rank as most underserved?\nGreen = Robust | Yellow = Moderate | Red = Sensitive")
ax.set_xlim(0, len(weight_schemes))
ax.axvline(4, color="green", linestyle="--", alpha=0.5, label="Robust threshold (4/5)")
ax.legend()
plt.tight_layout()
plt.show()

# Log to MLflow
if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name="Sensitivity_Analysis_Mismatch"):
        mlflow.set_tag("analysis_type", "sensitivity")
        mlflow.set_tag("test_year", str(test_year))
        
        # Log frequency of most stable country
        mlflow.log_metric("most_stable_country_freq", most_stable[1])
        mlflow.log_metric("n_schemes_tested", len(weight_schemes))
        mlflow.log_param("most_stable_country", most_stable[0])
        
        # Log plot
        with tempfile.TemporaryDirectory() as tmpdir:
            fig.savefig(f"{tmpdir}/sensitivity_analysis.png", dpi=150, bbox_inches="tight")
            mlflow.log_artifact(f"{tmpdir}/sensitivity_analysis.png", "plots")
        
        print("\n  ✅ Sensitivity analysis logged to MLflow")

In [ ]:
# ============================================================================
# CELL 5: Challenge 1 Models with MLflow (REPLACE CELL 30 - PART 1)
# ============================================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

MIN_BEN_REVIEW = 50

feat_df = model_df.dropna(subset=["budget_usd", "beneficiaries_total", "log10_cpb"]).copy()
feat_df_eff = feat_df[feat_df["beneficiaries_total"] >= MIN_BEN_REVIEW].copy()

# Use filtered dataset (beneficiaries >= 50)
df = feat_df_eff.copy()
df["log10_budget"] = np.log10(df["budget_usd"].where(df["budget_usd"] > 0))
df["log10_beneficiaries"] = np.log10(df["beneficiaries_total"].where(df["beneficiaries_total"] > 0))

target = df["log10_cpb"]

num_features = [
    "log10_budget",
    "log10_beneficiaries",
    "support_cost_share",
    "duration_months",
    "beneficiary_share_of_country_pop",
]
cat_features = [
    "cluster_primary",
    "OrganizationType",
    "AllocationTypeCategory",
]

num_features = [c for c in num_features if c in df.columns]
cat_features = [c for c in cat_features if c in df.columns]

X = df[num_features + cat_features]
X_train, X_test, y_train, y_test = train_test_split(X, target, test_size=0.25, random_state=42)

# Preprocessing
pre = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
            num_features,
        ),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore"))]),
            cat_features,
        ),
    ]
)

# ============================================================================
# MODEL 3: Ridge (Challenge 1) with MLflow
# ============================================================================

pipe_ridge_cpb = Pipeline([("pre", pre), ("model", Ridge(alpha=1.0))])
pipe_ridge_cpb.fit(X_train, y_train)

# Extract feature names
feature_names_cpb = list(num_features)
if cat_features:
    ohe = pipe_ridge_cpb.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_cpb.extend(ohe.get_feature_names_out(cat_features).tolist())

# Compute metrics
pred_train_cpb = pipe_ridge_cpb.predict(X_train)
pred_test_cpb = pipe_ridge_cpb.predict(X_test)

metrics_ridge_cpb = {
    "train_r2": r2_score(y_train, pred_train_cpb),
    "train_mae": mean_absolute_error(y_train, pred_train_cpb),
    "test_r2": r2_score(y_test, pred_test_cpb),
    "test_mae": mean_absolute_error(y_test, pred_test_cpb),
}

print(f"Ridge Regression (Challenge 1 - CPB, beneficiaries>={MIN_BEN_REVIEW}):")
print(f"  Train R²: {metrics_ridge_cpb['train_r2']:.4f} | MAE: {metrics_ridge_cpb['train_mae']:.4f}")
print(f"  Test  R²: {metrics_ridge_cpb['test_r2']:.4f} | MAE: {metrics_ridge_cpb['test_mae']:.4f}")

# Feature importance plot
coefs = pipe_ridge_cpb.named_steps["model"].coef_
coef_df = pd.DataFrame({"feature": feature_names_cpb, "coef": coefs})
coef_df["abs_coef"] = coef_df["coef"].abs()
coef_df = coef_df.sort_values("abs_coef", ascending=False)

plot_coef = coef_df.head(18).sort_values("coef")
fig_ridge_cpb, ax = plt.subplots(figsize=(10, 7))
colors = ["#dc2626" if v < 0 else "#16a34a" for v in plot_coef["coef"]]
ax.barh(plot_coef["feature"], plot_coef["coef"], color=colors, edgecolor="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Ridge: Factors associated with log10(CPB) (Challenge 1)")
ax.set_xlabel("Ridge coefficient (standardized)")
plt.tight_layout()

# Log to MLflow
if MLFLOW_AVAILABLE:
    with mlflow.start_run(run_name="Challenge1_Ridge_CPB_v1"):
        mlflow.set_tags({
            "challenge": "beneficiary_targeting",
            "model_type": "Ridge",
            "min_beneficiaries": str(MIN_BEN_REVIEW),
        })
        
        mlflow.log_params({
            "alpha": 1.0,
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_features": len(num_features + cat_features),
            "min_beneficiaries": MIN_BEN_REVIEW,
        })
        
        mlflow.log_metrics(metrics_ridge_cpb)
        mlflow.sklearn.log_model(pipe_ridge_cpb, "model")
        
        with tempfile.TemporaryDirectory() as tmpdir:
            fig_ridge_cpb.savefig(f"{tmpdir}/feature_importance.png", dpi=150, bbox_inches="tight")
            mlflow.log_artifact(f"{tmpdir}/feature_importance.png", "plots")
        
        print("  ✅ Logged to MLflow")

plt.show()

In [ ]:
# ============================================================================
# CELL 6: Challenge 1 XGBoost with MLflow (REPLACE CELL 30 - PART 2)
# ============================================================================

# XGBoost/GradientBoosting
try:
    from xgboost import XGBRegressor
    tree_model_cpb = XGBRegressor(
        n_estimators=350,
        max_depth=4,
        learning_rate=0.06,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42,
    )
    tree_model_name_cpb = "XGBoost"
    tree_ohe_sparse = True
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    tree_model_cpb = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
    tree_model_name_cpb = "GradientBoosting"
    tree_ohe_sparse = False

try:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=tree_ohe_sparse)
except TypeError:
    tree_ohe = OneHotEncoder(handle_unknown="ignore", sparse=tree_ohe_sparse)

pre_tree = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_features),
        (
            "cat",
            Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", tree_ohe)]),
            cat_features,
        ),
    ]
)

pipe_tree_cpb = Pipeline([("pre", pre_tree), ("model", tree_model_cpb)])
pipe_tree_cpb.fit(X_train, y_train)

# Extract feature names
feature_names_tree_cpb = list(num_features)
if cat_features:
    ohe_tree = pipe_tree_cpb.named_steps["pre"].named_transformers_["cat"].named_steps["ohe"]
    feature_names_tree_cpb.extend(ohe_tree.get_feature_names_out(cat_features).tolist())

# Compute metrics
pred_train_tree_cpb = pipe_tree_cpb.predict(X_train)
pred_test_tree_cpb = pipe_tree_cpb.predict(X_test)

metrics_tree_cpb = {
    "train_r2": r2_score(y_train, pred_train_tree_cpb),
    "train_mae": mean_absolute_error(y_train, pred_train_tree_cpb),
    "test_r2": r2_score(y_test, pred_test_tree_cpb),
    "test_mae": mean_absolute_error(y_test, pred_test_tree_cpb),
}

print(f"\n{tree_model_name_cpb} (Challenge 1 - CPB):")
print(f"  Train R²: {metrics_tree_cpb['train_r2']:.4f} | MAE: {metrics_tree_cpb['train_mae']:.4f}")
print(f"  Test  R²: {metrics_tree_cpb['test_r2']:.4f} | MAE: {metrics_tree_cpb['test_mae']:.4f}")

# Feature importance plot
if hasattr(pipe_tree_cpb.named_steps["model"], "feature_importances_"):
    importances = pipe_tree_cpb.named_steps["model"].feature_importances_
    imp_df = pd.DataFrame({"feature": feature_names_tree_cpb, "importance": importances})
    imp_df = imp_df.sort_values("importance", ascending=False).head(12).sort_values("importance")
    
    fig_tree_cpb, ax = plt.subplots(figsize=(10, 6))
    ax.barh(imp_df["feature"], imp_df["importance"], color="#2563eb", edgecolor="black", linewidth=0.5)
    ax.set_title(f"{tree_model_name_cpb}: Feature importance (Challenge 1 - CPB)")
    ax.set_xlabel("Feature importance")
    plt.tight_layout()
    
    # Log to MLflow
    if MLFLOW_AVAILABLE:
        with mlflow.start_run(run_name=f"Challenge1_{tree_model_name_cpb}_CPB_v1"):
            mlflow.set_tags({
                "challenge": "beneficiary_targeting",
                "model_type": tree_model_name_cpb,
                "min_beneficiaries": str(MIN_BEN_REVIEW),
            })
            
            params = {
                "n_train": len(X_train),
                "n_test": len(X_test),
                "n_features": len(num_features + cat_features),
                "min_beneficiaries": MIN_BEN_REVIEW,
            }
            
            if hasattr(tree_model_cpb, "n_estimators"):
                params["n_estimators"] = tree_model_cpb.n_estimators
            if hasattr(tree_model_cpb, "max_depth"):
                params["max_depth"] = tree_model_cpb.max_depth
            if hasattr(tree_model_cpb, "learning_rate"):
                params["learning_rate"] = tree_model_cpb.learning_rate
            
            mlflow.log_params(params)
            mlflow.log_metrics(metrics_tree_cpb)
            mlflow.sklearn.log_model(pipe_tree_cpb, "model")
            
            with tempfile.TemporaryDirectory() as tmpdir:
                fig_tree_cpb.savefig(f"{tmpdir}/feature_importance.png", dpi=150, bbox_inches="tight")
                mlflow.log_artifact(f"{tmpdir}/feature_importance.png", "plots")
            
            print("  ✅ Logged to MLflow")
    
    plt.show()

# Model comparison
print("\n" + "="*60)
print("CHALLENGE 1 (CPB) MODEL COMPARISON")
print("="*60)
print(f"Ridge:         Test R² = {metrics_ridge_cpb['test_r2']:.4f} | MAE = {metrics_ridge_cpb['test_mae']:.4f}")
print(f"{tree_model_name_cpb}: Test R² = {metrics_tree_cpb['test_r2']:.4f} | MAE = {metrics_tree_cpb['test_mae']:.4f}")
print("="*60)

In [ ]:
# ============================================================================
# CELL 7: Final Summary (ADD AT END OF NOTEBOOK)
# ============================================================================

print("\n" + "="*80)
print("           COMPLETE MODEL COMPARISON SUMMARY")
print("="*80)

if MLFLOW_AVAILABLE:
    # Get all runs from experiment
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment:
        runs_df = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            order_by=["metrics.test_r2 DESC"],
        )
        
        if not runs_df.empty:
            # Select key columns
            display_cols = [
                "tags.challenge",
                "tags.model_type",
                "metrics.test_r2",
                "metrics.test_mae",
                "params.n_train",
                "params.n_test",
            ]
            
            available_cols = [c for c in display_cols if c in runs_df.columns]
            comparison = runs_df[available_cols].copy()
            
            # Clean column names
            comparison.columns = [
                c.replace("tags.", "").replace("params.", "").replace("metrics.", "")
                for c in comparison.columns
            ]
            
            print("\nAll Model Runs:")
            print(comparison.to_string(index=False))
            
            # Best by challenge
            print("\n" + "-"*80)
            print("BEST MODEL BY CHALLENGE (highest test R²):")
            print("-"*80)
            
            if "challenge" in comparison.columns and "test_r2" in comparison.columns:
                best_per_challenge = comparison.loc[comparison.groupby("challenge")["test_r2"].idxmax()]
                for _, row in best_per_challenge.iterrows():
                    print(f"  {row['challenge']:30} -> {row['model_type']:15} (R² = {row['test_r2']:.4f})")
        else:
            print("\nNo MLflow runs found. Models may not have been logged.")
    else:
        print("\nMLflow experiment not found.")
else:
    print("\nMLflow not available. Install with: %pip install mlflow")

print("\n" + "="*80)
print("To view detailed results in Databricks:")
print("  1. Click 'Experiments' in left sidebar")
print("  2. Select 'DSC_Datathon_2026_Humanitarian_Analytics'")
print("  3. Compare runs side-by-side")
print("\nLocally:")
print("  1. Run: mlflow ui")
print("  2. Open: http://localhost:5000")
print("="*80)

---

## Integration Checklist

After copying cells to `DSC_Datathon.ipynb`:

- [ ] Cell 0 (MLflow setup) inserted after imports
- [ ] Cell 1 (Widgets) inserted after imports
- [ ] Cell 2 (Widget filters) inserted after feature engineering (Cell 10)
- [ ] Cell 3 (Geo-Insight models) replaced Cell 19
- [ ] Cell 4 (Sensitivity analysis) inserted after Forgotten Crisis Index (Cell 22)
- [ ] Cell 5-6 (Challenge 1 models) replaced Cell 30
- [ ] Cell 7 (Summary) added at end of notebook
- [ ] Run full notebook to verify MLflow logging works
- [ ] Check MLflow UI for all 4 model runs + sensitivity analysis

---